# ECG F1 fine-tune (GPU) — lift weak-class macro F1

Runs `tools/finetune_ecg_f1.py`: continue-trains the weak ECG pathology models
(**STACH, SBRAD, 1AVB, LBBB**) from their ecglib pretrained weights, and **keeps
each one only if it beats baseline on F1** (not balanced accuracy), with ECG-domain
augmentation. This is the run that can actually move macro F1 — the older
`colab_ecg_finetune.ipynb` optimised balanced accuracy and *rejected* F1 wins
(e.g. STACH F1 0.58 → 0.85).

### Before you start
1. **Rebuild the lean zip** so Colab gets the latest script — on your machine:
   `python "Colab PFE/make_lean_zip.py"` → upload the produced `medical-platform.zip`
   to the **root** of your Google Drive. (The cells below verify the script is present.)
2. **Runtime → Change runtime type → T4 GPU.**
3. Run the cells top to bottom. If the runtime restarts, re-run cell 3 (installs are wiped).

### What comes back
Kept checkpoints write `<PATHOLOGY>.pt` to `My Drive/colab_pfe_outputs/ecg_finetuned/`,
and the final `=== RESULTS ===` block lists each kept model's **tuned threshold**.
Paste that whole block back to Claude — it'll wire the `.pt` files **and** their
thresholds into `ecg_pipeline.py`, then re-verify locally.

### Honest target
macro F1 ~0.73 → **~0.78–0.82** if it goes well — **not** 0.90. The threshold-tuning
ceiling is ~0.75 and SBRAD/1AVB are prevalence-limited. A sudden jump toward 0.90
would be a leakage red flag, not a win. The no-regression rule means any class that
doesn't genuinely improve simply keeps its current weights (zero downside).

In [ ]:
# 1 — Confirm the GPU. If this prints "NO GPU", set
# Runtime > Change runtime type > T4 GPU, then re-run.
import torch
print(torch.__version__,
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

In [ ]:
# 2 — Mount Drive and unpack the repo (the lean zip you just rebuilt + uploaded).
# This must contain the LATEST tools/finetune_ecg_f1.py — if you skipped the
# rebuild, Colab will run an old script. The flatten check handles a zip that
# nested an extra top-level folder.
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/medical-platform
!mkdir -p /content/medical-platform
!unzip -q "/content/drive/My Drive/medical-platform.zip" -d /content/medical-platform
import os
root = "/content/medical-platform"
if not os.path.exists(f"{root}/tools") and os.path.exists(f"{root}/medical-platform/tools"):
    root = f"{root}/medical-platform"
assert os.path.exists(f"{root}/tools/finetune_ecg_f1.py"), \
    "finetune_ecg_f1.py not in the zip — rebuild it: python 'Colab PFE/make_lean_zip.py'"
print("repo root:", root)
%cd $root

In [ ]:
# 3 — Install the ECG deps (exact ecglib pin) + Python 3.12 patch.
# RE-RUN THIS after any runtime restart/reconnect — pip installs are wiped, but the
# downloaded data + mounted Drive survive. (Skipping it was the "No module named
# 'wfdb' / ecglib" failure.)
!pip -q install ecglib==1.0.1 wfdb

# ecglib 1.0.1 has a stray `import imp` (removed in Python 3.12, which Colab runs)
# that it never actually uses. Comment it out so ecglib imports cleanly. find_spec
# locates the file WITHOUT importing the package (importing is what crashes).
import importlib.util, pathlib, re
spec = importlib.util.find_spec("ecglib")
cfg = pathlib.Path(spec.origin).parent / "models" / "config" / "model_configs.py"
src = cfg.read_text()
if re.search(r"^import imp\b", src, flags=re.M):
    src = re.sub(r"^import imp\b.*$",
                 "# import imp  (removed in Py3.12; ecglib never uses it)",
                 src, count=1, flags=re.M)
    cfg.write_text(src); print("patched ecglib for Python 3.12")
else:
    print("ecglib already patched")

import wfdb, ecglib
print("ecglib + wfdb ready | wfdb", wfdb.__version__)

In [ ]:
# 4 — Download PTB-XL (official single zip ~1.7 GB, one connection).
# Sources, fastest first: already-extracted on this VM -> ptbxl.zip on your Drive
# -> PhysioNet download. Recursive wget (~43k requests) gets throttled — never use it.
import os, shutil
PTBXL_DIR = "physionet.org/files/ptb-xl/1.0.3"
ZIP_URL = ("https://physionet.org/static/published-projects/ptb-xl/"
           "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip")
DRIVE_ZIPS = ["/content/drive/My Drive/ptbxl.zip",
              "/content/drive/My Drive/ptb-xl-a-large-publicly-available-"
              "electrocardiography-dataset-1.0.3.zip"]
ZIP_PATH, EXTRACT_DIR = "/content/ptbxl.zip", "/content/ptbxl_extract"

def ptbxl_complete():
    if not os.path.exists(f"{PTBXL_DIR}/ptbxl_database.csv"):
        return False
    return sum(len(fs) for _, _, fs in os.walk(f"{PTBXL_DIR}/records500")) >= 43000

if not ptbxl_complete():
    shutil.rmtree(PTBXL_DIR, ignore_errors=True)            # wipe any partial tree
    if not os.path.exists(ZIP_PATH):
        dz = next((z for z in DRIVE_ZIPS if os.path.exists(z)), None)
        if dz:
            print("copying PTB-XL zip from Drive (no download)..."); shutil.copy(dz, ZIP_PATH)
        else:
            !wget -c --progress=bar:force:noscroll "{ZIP_URL}" -O "{ZIP_PATH}"
    print("unzipping (~2 min, silent)...")
    !unzip -q -o "{ZIP_PATH}" -d "{EXTRACT_DIR}"
    src = f"{EXTRACT_DIR}/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"
    if not os.path.exists(f"{src}/ptbxl_database.csv"):
        src = next(d for d, _, fs in os.walk(EXTRACT_DIR) if "ptbxl_database.csv" in fs)
    os.makedirs(os.path.dirname(PTBXL_DIR), exist_ok=True)
    shutil.move(src, PTBXL_DIR); shutil.rmtree(EXTRACT_DIR, ignore_errors=True)
    os.remove(ZIP_PATH)                                     # free 1.7 GB
    shutil.rmtree(f"{PTBXL_DIR}/records100", ignore_errors=True)  # 100 Hz copies unused

assert ptbxl_complete(), f"PTB-XL incomplete at {PTBXL_DIR} — re-run this cell"
print("PTB-XL ready at", PTBXL_DIR)

In [ ]:
# 5 — Fine-tune the weak ECG classes for F1 (augmentation ON).
# Keeps each checkpoint ONLY if it beats baseline on F1 (not balanced accuracy),
# and prints each kept model's tuned threshold at the end. ~30-60 min per class
# on a T4, plus a one-time ~10-15 min PTB-XL preprocessing pass (cached after).
!python tools/finetune_ecg_f1.py \
    --ptbxl-dir physionet.org/files/ptb-xl/1.0.3 \
    --pathologies STACH SBRAD 1AVB LBBB \
    --epochs 20 --augment \
    --cache-dir /content/ecg_ft_cache \
    --out-dir "/content/drive/My Drive/colab_pfe_outputs/ecg_finetuned"

# >>> Paste the printed "=== RESULTS ===" block back to Claude.
# It now includes the THRESHOLD for each kept checkpoint, so Claude can wire both
# the .pt files AND their thresholds into ecg_pipeline.py. Add --focal-gamma 2.0
# to try focal loss; omit --pathologies to run all 7.